# Kris — Qwen LoRA training on Colab

Trains the **same config as `configs/train.yaml`** (`Qwen/Qwen3.5-0.8B`, LoRA r=16, lr=2e-4, 3 epochs) but on a **Colab GPU** instead of local CPU.

**Before running:** set Runtime → Change runtime type → **T4 GPU** (or L4/A100 if available).

**Steps:** 1) GPU check → 2) clone + install → 3) HF login → 4) config → 5) train → 6) download adapter → 7) inference test.

## 1. GPU check
Run this first. You should see a Tesla T4 (or better) and `cuda: True`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

## 2. Clone repo + install deps

In [ ]:
# Clone the project (public repo — no token needed)
![ -d kris-model ] || git clone https://github.com/kamaravichow/kris-model.git
%cd kris-model
!git pull --ff-only

# Colab already ships torch; install the rest quietly
!pip install -q transformers>=4.56 trl>=0.12 peft>=0.10 datasets>=3.0 accelerate>=0.24 huggingface_hub>=0.23 python-dotenv pyyaml

import torch, transformers, trl, peft
print('deps:', torch.__version__, transformers.__version__, trl.__version__)

## 3. Hugging Face login
`Qwen/Qwen3.5-0.8B` is gated — request access on its model page once, then paste a **read** token below (never hard-code tokens in a notebook).

In [ ]:
from getpass import getpass
from huggingface_hub import login

tok = getpass('HF token (hf_...): ')
login(token=tok, add_to_git_credential=False)
print('HF login OK')

## 4. Config (edit me)
Defaults mirror `configs/train.yaml`. Only override what you need.

In [ ]:
import yaml

CFG_PATH = 'configs/train.yaml'   # repo default
cfg = yaml.safe_load(open(CFG_PATH))

# ---- overrides (edit these) ----
cfg['model_id'] = 'Qwen/Qwen3.5-0.8B'
cfg['data_path'] = 'data/filler.jsonl'   # 120 examples in repo
cfg['out_dir'] = 'outputs/kris-sft'
cfg['epochs'] = 3
cfg['lr'] = 2.0e-4
cfg['batch'] = 1          # per-device batch (T4-safe with grad_accum=8)
cfg['grad_accum'] = 8
cfg['max_len'] = 1024
cfg['hub_id'] = ''        # e.g. 'YOUR_USER/kris-sft' to push, '' = local only

print(yaml.safe_dump(cfg))

In [ ]:
# Sanity-check dataset (same checks as train.sh step 3/7)
import json
from pathlib import Path

p = Path(cfg['data_path'])
assert p.is_file(), f'dataset missing: {p} — git pull may have failed, or upload it manually'
rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
print(f'dataset lines: {len(rows)}')
print('roles:', [m['role'] for m in rows[0]['messages']])
print('preview:', json.dumps(rows[0])[:300])

## 5. Train (GPU)
Same LoRA + SFT logic as `src/train.py`, but **CUDA-enabled**: `device_map='auto'`, bf16/fp16 mixed precision, no `use_cpu`. ~45 optimizer steps (120 ex × 3 epochs ÷ eff. batch 8) — a few minutes on a T4.

In [ ]:
import datetime
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTTrainer, SFTConfig
from src.data import load_raw, to_text

def log(msg):
    print(f"[{datetime.datetime.now():%H:%M:%S}] {msg}", flush=True)

class StepPrintCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log(f"[train step {state.global_step}/{state.max_steps}] "
                f"loss={logs.get('loss', '?')} lr={logs.get('learning_rate', '?')} "
                f"grad_norm={logs.get('grad_norm', '?')}")
    def on_save(self, args, state, control, **kwargs):
        log(f'[checkpoint] saved at step {state.global_step}')

set_seed(42)
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU and re-run from §1'
BF16 = torch.cuda.is_bf16_supported()
log(f'device: {torch.cuda.get_device_name(0)} | bf16={BF16}')

log('[1/4] tokenizer...')
tok = AutoTokenizer.from_pretrained(cfg['model_id'], use_fast=True)
tok.pad_token = tok.pad_token or tok.eos_token
tok.padding_side = 'right'

log('[2/4] dataset...')
ds = to_text(load_raw(cfg), tok, cfg)
log(f'  examples: {len(ds)} | sample chars: {len(ds[0]["text"])}')

log('[3/4] model on CUDA...')
lora = LoraConfig(r=cfg['lora_r'], lora_alpha=cfg['lora_alpha'],
                  lora_dropout=cfg['lora_dropout'], bias='none',
                  task_type='CAUSAL_LM', target_modules=cfg['targets'])
model = AutoModelForCausalLM.from_pretrained(
    cfg['model_id'],
    dtype=torch.bfloat16 if BF16 else torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

log('[4/4] trainer...')
args = SFTConfig(
    output_dir=cfg['out_dir'],
    per_device_train_batch_size=cfg['batch'],
    gradient_accumulation_steps=cfg['grad_accum'],
    learning_rate=float(cfg['lr']),
    num_train_epochs=cfg['epochs'],
    max_length=cfg['max_len'],
    dataset_text_field='text',
    gradient_checkpointing=True,
    fp16=not BF16, bf16=BF16,
    logging_steps=1, logging_first_step=True,
    save_steps=200, save_total_limit=2,
    report_to='none',
    push_to_hub=bool(cfg.get('hub_id')), hub_model_id=cfg.get('hub_id') or None,
)
trainer = SFTTrainer(model=model, train_dataset=ds, peft_config=lora,
                     args=args, processing_class=tok, callbacks=[StepPrintCallback()])
total = len(trainer.get_train_dataloader()) * int(cfg['epochs'])
log(f'trainer ready: {len(ds)} ex, ~{total} steps '
    f"(batch={cfg['batch']} x accum={cfg['grad_accum']} x epochs={cfg['epochs']})")

trainer.train()
trainer.save_model(cfg['out_dir'])
log(f"saved: {cfg['out_dir']}")

## 6. Download the adapter
The LoRA adapter is small (~10–30 MB). This zips it for download.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

out = Path(cfg['out_dir'])
print('contents:', [p.name for p in sorted(out.iterdir())])
z = shutil.make_archive('kris-sft-adapter', 'zip', root_dir=out.parent, base_dir=out.name)
print('zip:', z, f"({Path(z).stat().st_size / 1e6:.1f} MB)")
files.download(z)

## 7. Inference test (base + adapter)

In [ ]:
import os, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base, adapter = cfg['model_id'], cfg['out_dir']
PROMPT = 'Clean fillers: So, um, I think we should, like, start the meeting now.'

tok = AutoTokenizer.from_pretrained(base, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    base, dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto', trust_remote_code=True)
if os.path.isdir(adapter):
    model = PeftModel.from_pretrained(model, adapter)
    print('adapter loaded:', adapter)

msgs = [{'role': 'user', 'content': PROMPT}]
inputs = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                 return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=256, do_sample=True,
                         temperature=0.7, top_p=0.9)
print(tok.decode(out[0], skip_special_tokens=True))

## Notes

- **OOM on T4?** Lower `cfg['batch']` to 1 (already default) or `cfg['max_len']` to 512 and re-run §5.
- **Push to Hub instead of downloading:** set `cfg['hub_id'] = 'YOUR_USER/kris-sft'` in §4 before training.
- **Local inference:** unzip the adapter next to a local clone and run `MODEL=Qwen/Qwen3.5-0.8B ADAPTER=outputs/kris-sft PROMPT='...' python src/infer.py`.
- `train_mlx.sh` / Apple-Silicon path is intentionally not covered here — Colab is CUDA-only.